# CV Lab 5

## 1)  Make Trackbar for the following images to tune the parameters of canny detection.

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt 

def nothing(x):
    pass

cv2.namedWindow('controller')

cv2.createTrackbar('threshold1', 'controller', 0, 1000, nothing)
cv2.createTrackbar('threshold2', 'controller', 0, 1000, nothing)
cv2.createTrackbar('choice', 'controller', 0, 1, nothing)

images = [cv2.imread('solidWhiteCurve.jpg') , cv2.imread('solidYellowCurve2.jpg')]
images1 = [cv2.cvtColor(images[0], cv2.COLOR_BGR2GRAY) , cv2.cvtColor(images[1], cv2.COLOR_BGR2GRAY)]
edges = [cv2.imread('solidWhiteCurve.jpg') , cv2.imread('solidYellowCurve2.jpg')]

while True:
    
    threshold1 = cv2.getTrackbarPos('threshold1', 'controller')
    threshold2 = cv2.getTrackbarPos('threshold2', 'controller')
    choice = cv2.getTrackbarPos('choice', 'controller')
    
    if(choice == 0):
        edges[0] = cv2.Canny(images1[0], threshold1, threshold2)
    elif(choice == 1):
        edges[1] = cv2.Canny(images1[1], threshold1, threshold2)
    
    cv2.imshow('controller', edges[choice])
    key = cv2.waitKey(10)
    if key == ord('q'):
        break

        
cv2.destroyAllWindows()

## Observations on Q1 : 
## - increasing parameters threshold1 and threshold2 removes unnecessary edges detected from the photos but increasing them too much will make them disappear completely
## - after some trials its found that : 
##           - best parameters for the first image (610,400)
##           - best parameters for the second image (430,400)
## - it needs only to apply region of interest for detecting only lanelines

## 2) The out of question 1 make it as input for question 2 after make region of interest then make trackbar for houghlines parameters (threshold ,min_line_length ,max_line_gap) and then draw 2 line on image.


In [2]:
import cv2
import numpy as np
import matplotlib.pyplot as plt 

def nothing(x):
    pass

def roi(edges):
    mask = np.zeros_like(edges)
    verts = np.array(
        [
            [
                (edges.shape[1]/12,edges.shape[0]),
                (edges.shape[1]*5/12,edges.shape[0]*3/5),
                (edges.shape[1]*7/12,edges.shape[0]*3/5),
                (edges.shape[1]*11/12 ,  edges.shape[0])
            ]
        ],np.int32
    )
    
    cv2.fillPoly(mask,verts,255)
    masked = cv2.bitwise_and(edges,mask)
    return masked

def houghLines(masked , threshold, min_line_length, max_line_gap):
    lines = cv2.HoughLinesP(masked,3.5,np.pi / 180, threshold = threshold, minLineLength = min_line_length, maxLineGap = max_line_gap)
    return lines

def form_lanes(lines,img):
    positive_slope =[]
    negative_slope =[]
    
    positive_inter =[]
    negative_inter =[]
    
    y_min = img.shape[0]
    y_max = img.shape[0]
    
    for line in lines:
        for x1,y1,x2,y2 in line :
            slope = (y2 - y1) / (x2 - x1)
            intercept = y2 -(slope*x2)
            y_min = min(y1,y2,y_min)
            
            if slope >= 0:
                positive_slope.append(slope)
                positive_inter.append(intercept)
                
            elif slope < 0:
                negative_slope.append(slope)
                negative_inter.append(intercept)
    
    positiveSlope = np.mean(positive_slope)
    negativeSlope = np.mean(negative_slope)
    
    positiveinter = np.mean(positive_inter)
    negativeinter = np.mean(negative_inter)
    
    pts = [
        [[0,0,0,0]],
        [[0,0,0,0]]
    ]
    
    if len(positive_slope) > 0:
        x_max = (y_max - positiveinter)/ positiveSlope
        x_min = (y_min - positiveinter)/ positiveSlope
        pts[0][0]= [x_min,y_min,x_max,y_max]
        
    if len(negative_slope) > 0:
        x_max = (y_max - negativeinter)/ negativeSlope
        x_min = (y_min - negativeinter)/ negativeSlope
        pts[1][0]= [x_min,y_min,x_max,y_max] 
    
    return np.array(pts,np.int32)

def drawLinesImg(img, lines):
    for line in lines:
          for x1,y1,x2,y2 in line:
                  cv2.line(img, (x1,y1), (x2,y2), [0,0,255], 10)
    return img

cv2.namedWindow('controller')

cv2.createTrackbar('threshold', 'controller', 8, 100, nothing)
cv2.createTrackbar('min_line_length', 'controller', 2, 20, nothing)
cv2.createTrackbar('max_line_gap', 'controller', 8, 100, nothing)
cv2.createTrackbar('choice', 'controller', 0, 1, nothing)

images = [cv2.imread('solidWhiteCurve.jpg') , cv2.imread('solidYellowCurve2.jpg')]
temp = images.copy()
edgesT = edges.copy()

while True:
    
    threshold = cv2.getTrackbarPos('threshold', 'controller')
    min_line_length = cv2.getTrackbarPos('min_line_length', 'controller')
    max_line_gap = cv2.getTrackbarPos('max_line_gap', 'controller')
    choice = cv2.getTrackbarPos('choice', 'controller')

    imagesTemp = temp[choice].copy()
        
    masked = roi(edgesT[choice])
    hough_lines = houghLines(masked ,threshold, min_line_length, max_line_gap )
    lanes = form_lanes(hough_lines , imagesTemp)
    result = drawLinesImg(imagesTemp,lanes)
    
    cv2.imshow('controller', result)
    key = cv2.waitKey(10)
    if key == ord('q'):
        break

        
cv2.destroyAllWindows()

## Observations on Q2 :
## - increasing threshold makes better line detection in terms of accuracy 
## - increasing min_line_length makes better line angle detection 
## - increasing max_line_gap removes much details and lines 
## - after some trials its found that : 
## -       best parameters for the first image (30,10,10)
## -       best parameters for the second image (32,4,7)

## 3) read the following video and apply lane line detection on it then show output using opencv.

In [3]:
import cv2
import numpy as np 
import matplotlib.pyplot as plt 

def toGray(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def findEdges(img,blurred):
    if blurred :
        img = cv2.GaussianBlur(img,(9,9),0)
    return cv2.Canny(img,60,160)

def roi(edges):
    mask = np.zeros_like(edges)
    verts = np.array(
        [
            [
                (edges.shape[1]/12,edges.shape[0]),
                (edges.shape[1]*5/12,edges.shape[0]*3/5),
                (edges.shape[1]*7/12,edges.shape[0]*3/5),
                (edges.shape[1]*11/12 ,  edges.shape[0])
            ]
        ],np.int32
    )
    cv2.fillPoly(mask,verts,255)
    masked = cv2.bitwise_and(edges,mask)
    return masked

def houghLines(masked):
    lines = cv2.HoughLinesP(masked,3.5,np.pi / 180,30,8,13)
    return lines

def lines_img_form(lines):
    lines_img = np.zeros((masked.shape[0],masked.shape[1],3),dtype=np.uint8)
    for line in lines:
        for x1,y1,x2,y2 in line :
            cv2.line(lines_img ,(x1,y1),(x2,y2),(0,0,255),10)
    return lines_img

def form_lanes(lines,img):
    positive_slope =[]
    negative_slope =[]
    
    positive_inter =[]
    negative_inter =[]
    
    y_min = img.shape[0]
    y_max = img.shape[0]
    
    for line in lines:
        for x1,y1,x2,y2 in line :
            slope = (y2 - y1) / (x2 - x1)
            intercept = y2 -(slope*x2)
            y_min = min(y1,y2,y_min)
            
            if slope >= 0.0:
                positive_slope.append(slope)
                positive_inter.append(intercept)
                
            elif slope < 0.0:
                negative_slope.append(slope)
                negative_inter.append(intercept)
    
    positiveSlope = np.mean(positive_slope)
    negativeSlope = np.mean(negative_slope)
    
    positiveinter = np.mean(positive_inter)
    negativeinter = np.mean(negative_inter)
    
    pts = [
        [[0,0,0,0]],
        [[0,0,0,0]]
    ]
    
    if len(positive_slope) > 0:
        x_max = (y_max - positiveinter)/ positiveSlope
        x_min = (y_min - positiveinter)/ positiveSlope
        pts[0][0]= [x_min,y_min,x_max,y_max]
        
    if len(negative_slope) > 0:
        x_max = (y_max - negativeinter)/ negativeSlope
        x_min = (y_min - negativeinter)/ negativeSlope
        pts[1][0]= [x_min,y_min,x_max,y_max] 
    
    return np.array(pts,np.int32)

cap = cv2.VideoCapture('solidWhiteRight.mp4')

while(cap.isOpened()):
    state,frame = cap.read()
    if state == True:
        gray = toGray(frame)
        edges = findEdges(gray,1)
        masked = roi(edges)
        hough_lines = houghLines(masked)
        lanes = form_lanes(hough_lines , gray)
        lines_img =lines_img_form(lanes)        
        result = cv2.addWeighted(frame, .9, lines_img, 1, 0)
        
        cv2.imshow('frame',result)
        key = cv2.waitKey(30)
        if key == ord('q'):
            break
    else:
        break

cv2.destroyAllWindows()
cap.release()